<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/4_1_Decadal_Land_Use_Land_Cover_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =============================================================================
# TEZPUR–NORTH LAKHIMPUR HIGHWAY CORRIDOR
# LULC STATISTICS FROM DYNAMIC WORLD GEOTIFF FILES
# =============================================================================
#
# Purpose:
#   Calculate annual LULC area and percentage statistics from exported
#   Dynamic World GeoTIFF files.
#
# Required TIFF files:
#   TZPR_NLP_LULC_2016.tif
#   TZPR_NLP_LULC_2018.tif
#   TZPR_NLP_LULC_2020.tif
#   TZPR_NLP_LULC_2022.tif
#   TZPR_NLP_LULC_2024.tif
#   TZPR_NLP_LULC_2025.tif
#   TZPR_NLP_LULC_2026_YTD.tif   [optional]
#
# Output:
#   1. LULC area in hectares
#   2. LULC area in km2
#   3. LULC percentage
#   4. LULC change between 2016 and 2025
#   5. Percentage change
#   6. CSV tables
#   7. Excel workbook
#   8. Manuscript-ready summary text
#
# Dynamic World classes:
#   0 = Water
#   1 = Trees
#   2 = Grass
#   3 = Flooded vegetation
#   4 = Crops
#   5 = Shrub & scrub
#   6 = Built
#   7 = Bare
#   8 = Snow & ice
#
# =============================================================================


import os
import glob
import numpy as np
import pandas as pd
import rasterio


# =============================================================================
# 1. USER SETTINGS
# =============================================================================

# -------------------------------------------------------------------------
# CHANGE THIS TO YOUR TIFF FOLDER
# -------------------------------------------------------------------------

INPUT_FOLDER = r"/content/drive/MyDrive/TZPR_NLP_Research"

# -------------------------------------------------------------------------
# OUTPUT FOLDER
# -------------------------------------------------------------------------

OUTPUT_FOLDER = os.path.join(
    INPUT_FOLDER,
    "LULC_Statistics"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# =============================================================================
# 2. DYNAMIC WORLD CLASS INFORMATION
# =============================================================================

CLASS_NAMES = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded Vegetation",
    4: "Crops",
    5: "Shrub",
    6: "Built",
    7: "Bare",
    8: "Snow"
}


# =============================================================================
# 3. EXPECTED TIFF FILES
# =============================================================================

FILES = {

    2016: "TZPR_NLP_LULC_2016.tif",

    2018: "TZPR_NLP_LULC_2018.tif",

    2020: "TZPR_NLP_LULC_2020.tif",

    2022: "TZPR_NLP_LULC_2022.tif",

    2024: "TZPR_NLP_LULC_2024.tif",

    2025: "TZPR_NLP_LULC_2025.tif",

    2026: "TZPR_NLP_LULC_2026_YTD.tif"
}


# =============================================================================
# 4. FIND FILES
# =============================================================================

print("\n" + "=" * 80)
print("TEZPUR–NORTH LAKHIMPUR LULC STATISTICS")
print("=" * 80)

available_files = {}

for year, filename in FILES.items():

    filepath = os.path.join(
        INPUT_FOLDER,
        filename
    )

    if os.path.exists(filepath):

        available_files[year] = filepath

        print(
            f"[FOUND] {year}: {filename}"
        )

    else:

        print(
            f"[MISSING] {year}: {filename}"
        )


if len(available_files) == 0:

    raise FileNotFoundError(
        "\nNo LULC GeoTIFF files were found.\n"
        "Check INPUT_FOLDER in the script."
    )


# =============================================================================
# 5. FUNCTION TO READ LULC TIFF
# =============================================================================

def read_lulc_statistics(
    filepath,
    year
):

    print("\n" + "-" * 80)

    print(
        f"Processing LULC {year}"
    )

    print(
        f"File: {filepath}"
    )

    with rasterio.open(filepath) as src:

        data = src.read(1)

        transform = src.transform

        crs = src.crs

        nodata = src.nodata

        width = src.width
        height = src.height

        print(
            f"Raster size: {width} x {height}"
        )

        print(
            f"CRS: {crs}"
        )

        print(
            f"NoData: {nodata}"
        )


        # ---------------------------------------------------------------------
        # PIXEL AREA
        # ---------------------------------------------------------------------
        #
        # The GEE export uses:
        #
        #   scale = 10 m
        #
        # Therefore:
        #
        #   pixel area = 10 x 10 = 100 m²
        #
        #   hectares = 100 / 10,000 = 0.01 ha
        #
        # However, this script calculates pixel area from the GeoTIFF
        # transform so that it also works for slightly different resolutions.
        #
        # ---------------------------------------------------------------------

        pixel_width = abs(
            transform.a
        )

        pixel_height = abs(
            transform.e
        )

        pixel_area_m2 = (
            pixel_width *
            pixel_height
        )

        pixel_area_ha = (
            pixel_area_m2 /
            10000
        )

        pixel_area_km2 = (
            pixel_area_m2 /
            1_000_000
        )


        print(
            f"Pixel size: "
            f"{pixel_width:.3f} x "
            f"{pixel_height:.3f} m"
        )

        print(
            f"Pixel area: "
            f"{pixel_area_m2:.3f} m²"
        )


        # ---------------------------------------------------------------------
        # VALID PIXELS
        # ---------------------------------------------------------------------

        if nodata is not None:

            valid_mask = (
                data != nodata
            )

        else:

            valid_mask = np.isfinite(
                data
            )


        valid_data = data[
            valid_mask
        ]


        # ---------------------------------------------------------------------
        # CLASS STATISTICS
        # ---------------------------------------------------------------------

        rows = []

        total_pixels = len(
            valid_data
        )

        total_area_ha = (
            total_pixels *
            pixel_area_ha
        )

        total_area_km2 = (
            total_pixels *
            pixel_area_km2
        )


        print(
            f"Valid pixels: {total_pixels:,}"
        )

        print(
            f"Total area: "
            f"{total_area_ha:,.2f} ha"
        )

        print(
            f"Total area: "
            f"{total_area_km2:,.2f} km²"
        )


        for class_id, class_name in CLASS_NAMES.items():

            pixel_count = np.sum(
                valid_data == class_id
            )

            area_ha = (
                pixel_count *
                pixel_area_ha
            )

            area_km2 = (
                pixel_count *
                pixel_area_km2
            )

            percentage = (
                area_ha /
                total_area_ha *
                100
                if total_area_ha > 0
                else 0
            )


            rows.append({

                "Year":
                    year,

                "Class_ID":
                    class_id,

                "LULC_Class":
                    class_name,

                "Pixel_Count":
                    int(pixel_count),

                "Area_ha":
                    area_ha,

                "Area_km2":
                    area_km2,

                "Percentage":
                    percentage

            })


        return pd.DataFrame(
            rows
        )


# =============================================================================
# 6. PROCESS ALL YEARS
# =============================================================================

all_statistics = []

for year, filepath in sorted(
    available_files.items()
):

    df = read_lulc_statistics(
        filepath,
        year
    )

    all_statistics.append(
        df
    )


lulc_stats = pd.concat(
    all_statistics,
    ignore_index=True
)


# =============================================================================
# 7. ROUND VALUES
# =============================================================================

lulc_stats[
    "Area_ha"
] = lulc_stats[
    "Area_ha"
].round(3)


lulc_stats[
    "Area_km2"
] = lulc_stats[
    "Area_km2"
].round(3)


lulc_stats[
    "Percentage"
] = lulc_stats[
    "Percentage"
].round(3)


# =============================================================================
# 8. SAVE COMPLETE LONG-FORM TABLE
# =============================================================================

long_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_LULC_Complete_Statistics.csv"
)

lulc_stats.to_csv(
    long_csv,
    index=False
)


print(
    f"\nSaved:\n{long_csv}"
)


# =============================================================================
# 9. CREATE AREA TABLE
# =============================================================================

area_table = lulc_stats.pivot(
    index="Year",
    columns="LULC_Class",
    values="Area_ha"
)

area_table = area_table.reset_index()


area_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_LULC_Area_ha_by_Year.csv"
)

area_table.to_csv(
    area_csv,
    index=False
)


# =============================================================================
# 10. CREATE KM² TABLE
# =============================================================================

area_km2_table = lulc_stats.pivot(
    index="Year",
    columns="LULC_Class",
    values="Area_km2"
)

area_km2_table = (
    area_km2_table
    .reset_index()
)


km2_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_LULC_Area_km2_by_Year.csv"
)

area_km2_table.to_csv(
    km2_csv,
    index=False
)


# =============================================================================
# 11. CREATE PERCENTAGE TABLE
# =============================================================================

percentage_table = lulc_stats.pivot(
    index="Year",
    columns="LULC_Class",
    values="Percentage"
)

percentage_table = (
    percentage_table
    .reset_index()
)


percentage_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_LULC_Percentage_by_Year.csv"
)

percentage_table.to_csv(
    percentage_csv,
    index=False
)


# =============================================================================
# 12. 2016–2025 CHANGE ANALYSIS
# =============================================================================

if 2016 in available_files and 2025 in available_files:

    stats_2016 = lulc_stats[
        lulc_stats["Year"] == 2016
    ].set_index(
        "LULC_Class"
    )

    stats_2025 = lulc_stats[
        lulc_stats["Year"] == 2025
    ].set_index(
        "LULC_Class"
    )


    change_rows = []


    for class_name in CLASS_NAMES.values():

        area_2016 = stats_2016.loc[
            class_name,
            "Area_ha"
        ]

        area_2025 = stats_2025.loc[
            class_name,
            "Area_ha"
        ]

        pct_2016 = stats_2016.loc[
            class_name,
            "Percentage"
        ]

        pct_2025 = stats_2025.loc[
            class_name,
            "Percentage"
        ]


        absolute_change = (
            area_2025 -
            area_2016
        )


        percentage_change = (
            absolute_change /
            area_2016 *
            100
            if area_2016 != 0
            else np.nan
        )


        share_change = (
            pct_2025 -
            pct_2016
        )


        change_rows.append({

            "LULC_Class":
                class_name,

            "Area_2016_ha":
                area_2016,

            "Area_2025_ha":
                area_2025,

            "Absolute_Change_ha":
                absolute_change,

            "Percentage_Change":
                percentage_change,

            "Share_2016_percent":
                pct_2016,

            "Share_2025_percent":
                pct_2025,

            "Share_Change_percentage_points":
                share_change

        })


    change_table = pd.DataFrame(
        change_rows
    )


    change_table[
        "Area_2016_ha"
    ] = change_table[
        "Area_2016_ha"
    ].round(3)


    change_table[
        "Area_2025_ha"
    ] = change_table[
        "Area_2025_ha"
    ].round(3)


    change_table[
        "Absolute_Change_ha"
    ] = change_table[
        "Absolute_Change_ha"
    ].round(3)


    change_table[
        "Percentage_Change"
    ] = change_table[
        "Percentage_Change"
    ].round(3)


    change_table[
        "Share_2016_percent"
    ] = change_table[
        "Share_2016_percent"
    ].round(3)


    change_table[
        "Share_2025_percent"
    ] = change_table[
        "Share_2025_percent"
    ].round(3)


    change_table[
        "Share_Change_percentage_points"
    ] = change_table[
        "Share_Change_percentage_points"
    ].round(3)


    change_csv = os.path.join(
        OUTPUT_FOLDER,
        "TZPR_NLP_LULC_Change_2016_2025.csv"
    )


    change_table.to_csv(
        change_csv,
        index=False
    )


    print(
        f"\nSaved:\n{change_csv}"
    )


# =============================================================================
# 13. IDENTIFY MAJOR GAINS AND LOSSES
# =============================================================================

if 2016 in available_files and 2025 in available_files:

    print(
        "\n" + "=" * 80
    )

    print(
        "2016–2025 MAJOR LULC CHANGES"
    )

    print(
        "=" * 80
    )


    sorted_change = change_table.sort_values(
        "Absolute_Change_ha",
        ascending=False
    )


    print(
        "\nLargest increases:"
    )


    print(
        sorted_change[
            [
                "LULC_Class",
                "Absolute_Change_ha"
            ]
        ].head(5).to_string(
            index=False
        )
    )


    print(
        "\nLargest decreases:"
    )


    print(
        sorted_change[
            [
                "LULC_Class",
                "Absolute_Change_ha"
            ]
        ].tail(5).sort_values(
            "Absolute_Change_ha"
        ).to_string(
            index=False
        )
    )


# =============================================================================
# 14. BUILT-UP SPECIFIC STATISTICS
# =============================================================================

built_stats = lulc_stats[
    lulc_stats[
        "LULC_Class"
    ] == "Built"
].copy()


built_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_BuiltUp_Statistics_By_Year.csv"
)


built_stats.to_csv(
    built_csv,
    index=False
)


print(
    f"\nSaved:\n{built_csv}"
)


# =============================================================================
# 15. CROPLAND SPECIFIC STATISTICS
# =============================================================================

crop_stats = lulc_stats[
    lulc_stats[
        "LULC_Class"
    ] == "Crops"
].copy()


crop_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_Cropland_Statistics_By_Year.csv"
)


crop_stats.to_csv(
    crop_csv,
    index=False
)


# =============================================================================
# 16. TREE COVER SPECIFIC STATISTICS
# =============================================================================

tree_stats = lulc_stats[
    lulc_stats[
        "LULC_Class"
    ] == "Trees"
].copy()


tree_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_TreeCover_Statistics_By_Year.csv"
)


tree_stats.to_csv(
    tree_csv,
    index=False
)


# =============================================================================
# 17. VEGETATION GROUP
# =============================================================================
#
# Vegetation =
#
# Trees
# Grass
# Flooded Vegetation
# Crops
# Shrub
#
# =============================================================================

vegetation_classes = [
    "Trees",
    "Grass",
    "Flooded Vegetation",
    "Crops",
    "Shrub"
]


vegetation_stats = (
    lulc_stats[
        lulc_stats[
            "LULC_Class"
        ].isin(
            vegetation_classes
        )
    ]
    .groupby("Year")
    .agg(
        Vegetation_Area_ha=(
            "Area_ha",
            "sum"
        ),
        Vegetation_Area_km2=(
            "Area_km2",
            "sum"
        ),
        Vegetation_Percentage=(
            "Percentage",
            "sum"
        )
    )
    .reset_index()
)


vegetation_csv = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_Total_Vegetation_Statistics.csv"
)


vegetation_stats.to_csv(
    vegetation_csv,
    index=False
)


# =============================================================================
# 18. BUILT-UP EXPANSION 2016–2025
# =============================================================================

if 2016 in available_files and 2025 in available_files:

    built2016 = float(
        built_stats[
            built_stats["Year"] == 2016
        ]["Area_ha"].iloc[0]
    )

    built2025 = float(
        built_stats[
            built_stats["Year"] == 2025
        ]["Area_ha"].iloc[0]
    )


    built_absolute_change = (
        built2025 -
        built2016
    )


    built_percentage_change = (
        built_absolute_change /
        built2016 *
        100
        if built2016 != 0
        else np.nan
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "BUILT-UP EXPANSION"
    )

    print(
        "=" * 80
    )

    print(
        f"Built-up area 2016: "
        f"{built2016:.2f} ha"
    )

    print(
        f"Built-up area 2025: "
        f"{built2025:.2f} ha"
    )

    print(
        f"Absolute increase: "
        f"{built_absolute_change:.2f} ha"
    )

    print(
        f"Percentage increase: "
        f"{built_percentage_change:.2f}%"
    )


# =============================================================================
# 19. CROPLAND CHANGE
# =============================================================================

if 2016 in available_files and 2025 in available_files:

    crop2016 = float(
        crop_stats[
            crop_stats["Year"] == 2016
        ]["Area_ha"].iloc[0]
    )

    crop2025 = float(
        crop_stats[
            crop_stats["Year"] == 2025
        ]["Area_ha"].iloc[0]
    )


    crop_change = (
        crop2025 -
        crop2016
    )


    crop_percentage_change = (
        crop_change /
        crop2016 *
        100
        if crop2016 != 0
        else np.nan
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "CROPLAND CHANGE"
    )

    print(
        "=" * 80
    )

    print(
        f"Cropland area 2016: "
        f"{crop2016:.2f} ha"
    )

    print(
        f"Cropland area 2025: "
        f"{crop2025:.2f} ha"
    )

    print(
        f"Absolute change: "
        f"{crop_change:.2f} ha"
    )

    print(
        f"Percentage change: "
        f"{crop_percentage_change:.2f}%"
    )


# =============================================================================
# 20. TREE COVER CHANGE
# =============================================================================

if 2016 in available_files and 2025 in available_files:

    tree2016 = float(
        tree_stats[
            tree_stats["Year"] == 2016
        ]["Area_ha"].iloc[0]
    )

    tree2025 = float(
        tree_stats[
            tree_stats["Year"] == 2025
        ]["Area_ha"].iloc[0]
    )


    tree_change = (
        tree2025 -
        tree2016
    )


    tree_percentage_change = (
        tree_change /
        tree2016 *
        100
        if tree2016 != 0
        else np.nan
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "TREE COVER CHANGE"
    )

    print(
        "=" * 80
    )

    print(
        f"Tree area 2016: "
        f"{tree2016:.2f} ha"
    )

    print(
        f"Tree area 2025: "
        f"{tree2025:.2f} ha"
    )

    print(
        f"Absolute change: "
        f"{tree_change:.2f} ha"
    )

    print(
        f"Percentage change: "
        f"{tree_percentage_change:.2f}%"
    )


# =============================================================================
# 21. CREATE EXCEL WORKBOOK
# =============================================================================

excel_file = os.path.join(
    OUTPUT_FOLDER,
    "TZPR_NLP_LULC_Statistics_Complete.xlsx"
)


with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    lulc_stats.to_excel(
        writer,
        sheet_name="Complete_Statistics",
        index=False
    )

    area_table.to_excel(
        writer,
        sheet_name="Area_ha",
        index=False
    )

    area_km2_table.to_excel(
        writer,
        sheet_name="Area_km2",
        index=False
    )

    percentage_table.to_excel(
        writer,
        sheet_name="Percentage",
        index=False
    )

    built_stats.to_excel(
        writer,
        sheet_name="Built_Up",
        index=False
    )

    crop_stats.to_excel(
        writer,
        sheet_name="Cropland",
        index=False
    )

    tree_stats.to_excel(
        writer,
        sheet_name="Trees",
        index=False
    )

    vegetation_stats.to_excel(
        writer,
        sheet_name="Vegetation",
        index=False
    )

    if 2016 in available_files and 2025 in available_files:

        change_table.to_excel(
            writer,
            sheet_name="2016_2025_Change",
            index=False
        )


print(
    f"\nExcel workbook saved:\n{excel_file}"
)


# =============================================================================
# 22. PRINT MANUSCRIPT-READY TABLE
# =============================================================================

print(
    "\n" + "=" * 100
)

print(
    "MANUSCRIPT TABLE: LULC AREA AND PERCENTAGE"
)

print(
    "=" * 100
)


manuscript_table = lulc_stats[
    [
        "Year",
        "LULC_Class",
        "Area_ha",
        "Area_km2",
        "Percentage"
    ]
].copy()


print(
    manuscript_table.to_string(
        index=False
    )
)


# =============================================================================
# 23. GENERATE SIMPLE MANUSCRIPT SUMMARY
# =============================================================================

summary_file = os.path.join(
    OUTPUT_FOLDER,
    "Section_4_1_LULC_Statistics_Summary.txt"
)


with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "SECTION 4.1 – DECADAL LAND-USE/LAND-COVER DYNAMICS\n"
    )

    f.write(
        "=" * 80 + "\n\n"
    )


    if 2016 in available_files and 2025 in available_files:

        f.write(
            "2016–2025 CHANGE SUMMARY\n"
        )

        f.write(
            "-" * 80 + "\n"
        )


        for _, row in change_table.iterrows():

            f.write(
                f"{row['LULC_Class']}: "
                f"{row['Area_2016_ha']:.2f} ha "
                f"→ "
                f"{row['Area_2025_ha']:.2f} ha; "
                f"change = "
                f"{row['Absolute_Change_ha']:.2f} ha; "
                f"{row['Percentage_Change']:.2f}%\n"
            )


        f.write(
            "\n"
        )


    f.write(
        "YEARLY LULC AREA\n"
    )

    f.write(
        "-" * 80 + "\n\n"
    )


    for year in sorted(
        available_files.keys()
    ):

        f.write(
            f"\n{year}\n"
        )

        year_df = lulc_stats[
            lulc_stats["Year"] == year
        ]


        for _, row in year_df.iterrows():

            f.write(
                f"  {row['LULC_Class']}: "
                f"{row['Area_ha']:.2f} ha "
                f"({row['Percentage']:.2f}%)\n"
            )


print(
    f"\nManuscript summary saved:\n{summary_file}"
)


# =============================================================================
# 24. FINAL OUTPUT LIST
# =============================================================================

print(
    "\n" + "=" * 80
)

print(
    "PROCESSING COMPLETE"
)

print(
    "=" * 80
)

print(
    f"\nOutput folder:\n{OUTPUT_FOLDER}"
)

print(
    "\nGenerated files:"
)

print(
    "1. TZPR_NLP_LULC_Complete_Statistics.csv"
)

print(
    "2. TZPR_NLP_LULC_Area_ha_by_Year.csv"
)

print(
    "3. TZPR_NLP_LULC_Area_km2_by_Year.csv"
)

print(
    "4. TZPR_NLP_LULC_Percentage_by_Year.csv"
)

print(
    "5. TZPR_NLP_LULC_Change_2016_2025.csv"
)

print(
    "6. TZPR_NLP_BuiltUp_Statistics_By_Year.csv"
)

print(
    "7. TZPR_NLP_Cropland_Statistics_By_Year.csv"
)

print(
    "8. TZPR_NLP_TreeCover_Statistics_By_Year.csv"
)

print(
    "9. TZPR_NLP_Total_Vegetation_Statistics.csv"
)

print(
    "10. TZPR_NLP_LULC_Statistics_Complete.xlsx"
)

print(
    "11. Section_4_1_LULC_Statistics_Summary.txt"
)

print(
    "\n" + "=" * 80
)